# Day 2: RAG, step 1: turn IMD documents into chunks

## Step 1: Choose the documents and pages

In [1]:
from pathlib import Path

DOCS_DIR = Path("../docs/reference")

DOCUMENTS = [
    {
        "doc_id": "heat_faq",
        "file": "FAQ_heat_wave.pdf",
        "title": "IMD: FAQ on Heat Wave",
        "url": "https://mausam.imd.gov.in/pdfs/heatcolduser/FAQ_heat_wave.pdf",
        "pages": "all",
    },
    {
        "doc_id": "monsoon_faq",
        "file": "monsoonfaq.pdf",
        "title": "IMD: FAQ on Monsoon",
        "url": "https://mausam.imd.gov.in/imd_latest/monsoonfaq.pdf",
        "pages": "all",
    },
    {
        "doc_id": "cyclone_terms",
        "file": "terminology.pdf",
        "title": "IMD RSMC New Delhi: Cyclone terminology",
        "url": "https://rsmcnewdelhi.imd.gov.in/images/pdf/terminology.pdf",
        "pages": "all",
    },
    {
        "doc_id": "forecast_sop",
        "file": "forecasting_sop.pdf",
        "title": "IMD: SOP for Weather Forecasting and Warning",
        "url": "https://mausam.imd.gov.in/imd_latest/contents/pdf/forecasting_sop.pdf",
        "pages": [range(24, 28), range(171, 183)],
    },
]

for doc in DOCUMENTS:
    path = DOCS_DIR / doc["file"]
    print(f"{doc['doc_id']:14} {doc['file']:22} found: {path.exists()}")

heat_faq       FAQ_heat_wave.pdf      found: True
monsoon_faq    monsoonfaq.pdf         found: True
cyclone_terms  terminology.pdf        found: True
forecast_sop   forecasting_sop.pdf    found: True


## Step 2: Read the text, page by page

In [2]:
from pypdf import PdfReader

def pages_to_read(doc, total_pages):
    if doc["pages"] == "all":
        return list(range(1, total_pages + 1))
    return [p for page_range in doc["pages"] for p in page_range]

pages = []
for doc in DOCUMENTS:
    reader = PdfReader(DOCS_DIR / doc["file"])
    for page_no in pages_to_read(doc, len(reader.pages)):
        text = reader.pages[page_no - 1].extract_text() or ""
        pages.append({"doc_id": doc["doc_id"], "page": page_no, "text": text})

print(f"Read {len(pages)} pages in total")

Read 60 pages in total


### How much text did we get from each document?

In [3]:
import pandas as pd

page_table = pd.DataFrame(pages)
page_table["words"] = page_table["text"].str.split().str.len()

summary = page_table.groupby("doc_id").agg(
    pages=("page", "count"),
    words=("words", "sum"),
    empty_pages=("words", lambda w: (w == 0).sum()),
)
summary

,pages,words,empty_pages
doc_id,,,
cyclone_terms,8,2237,0
forecast_sop,16,5686,0
heat_faq,16,4092,0
monsoon_faq,20,8019,0


### Look at one raw page

In [4]:
first_page = next(p for p in pages if p["doc_id"] == "heat_faq" and p["page"] == 1)
print(first_page["text"][:1200])

1 | P a g e  
 
FAQ on Heat Wave 
 
Q. What is a heat wave? 
Heat Waves are a period of unusually high temperatures as compared to what is 
normally expected over a region. Therefore, the temperatures at which Heat waves are 
declared differ from place to place based on the te mperature climatology (historical 
temperatures) of that region. The impact of heat waves gets aggravated by supportive 
meteorological factors such as high humidity, high wind speed, duration of heat wave 
events, etc. 
 
Q. How to check Heatwave levels? 
India Meteorological Department provides color-coded Heat Wave warning information 
through its daily bulletins as well as through a GIS -based visualization platform for the 
next 5 days. The same can be seen at www.mausam.imd.gov.in Heat 
WaveGuidance. 
 
Q. Will I be affected by the Heatwave? 
You may be affected if the heat wave forecast map shows a heat wave warning over 
your region or regions close to you. In this scenario, all precautions to mitigate th

## Step 3: Clean the text

In [5]:
import re
from collections import Counter

HEADER_PATTERNS = [
    r"^\d+\s*\|\s*P a g e$",
    r"^SOP for Weather Forecasting and Warning\s*\d*$",
    r"^Heat and Cold Wave Monitoring & Warning Services\s*\d*$",
    r"^General Forecasting Organisation of IMD\s*\d*$",
]
NEVER_JOIN = {"or", "and", "the", "a", "an", "of", "in", "to", "per", "is", "on", "at",
              "as", "by", "be", "it", "rain", "wide", "frame", "sea", "day", "time"}

def remove_headers(text):
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if not any(re.match(p, line) for p in HEADER_PATTERNS)]
    while lines and (lines[0] == "" or lines[0].isdigit()):
        lines.pop(0)
    while lines and (lines[-1] == "" or lines[-1].isdigit()):
        lines.pop()
    return "\n".join(lines)

def remove_links(text):
    text = re.sub(r"https?://\S+", " ", text)
    return re.sub(r"\S*%[0-9A-F]{2}\S*", " ", text)

def fix_degrees(text):
    text = text.replace("℃", "°C").replace("º", "°")
    text = re.sub(r"(\d) 0C\b", r"\1°C", text)
    text = re.sub(r"(\d)0C\b", r"\1°C", text)
    text = re.sub(r"(?<![\w.%])(\d+(?:\.\d+)?)C\b", r"\1°C", text)
    return text

def fix_symbols(text):
    for bullet in ["", "", "♦", "•"]:
        text = text.replace(bullet, "-")
    for curly, plain in {"“": '"', "”": '"', "‘": "'", "’": "'"}.items():
        text = text.replace(curly, plain)
    return re.sub(r"(\w) -(\w)", r"\1-\2", text)

def build_vocabulary(texts):
    return Counter(w.lower() for t in texts for w in re.findall(r"[A-Za-z]+", t))

def should_join(a, b, vocab):
    is_word = vocab[(a + b).lower()] > 0
    has_fragment = min(vocab[a.lower()], vocab[b.lower()]) <= 2
    safe = a.lower() not in NEVER_JOIN and b.lower() not in NEVER_JOIN and len(a + b) >= 4
    return is_word and has_fragment and safe

def fix_split_words(text, vocab, joins):
    fixed_lines = []
    for line in text.split("\n"):
        words = line.split(" ")
        result, i = [], 0
        while i < len(words):
            if i + 1 < len(words):
                a = re.fullmatch(r"(\W*)([A-Za-z]+)", words[i])
                b = re.fullmatch(r"([a-z]+)(\W*)", words[i + 1])
                if a and b and should_join(a.group(2), b.group(1), vocab):
                    joins[f"{a.group(2)} {b.group(1)} -> {a.group(2) + b.group(1)}"] += 1
                    result.append(a.group(1) + a.group(2) + b.group(1) + b.group(2))
                    i += 2
                    continue
            result.append(words[i])
            i += 1
        fixed_lines.append(" ".join(result))
    return "\n".join(fixed_lines)

def squeeze_spaces(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

### Apply the cleaning to every page

In [6]:
vocab = build_vocabulary(p["text"] for p in pages)
joins = Counter()

for p in pages:
    text = remove_headers(p["text"])
    text = remove_links(text)
    text = fix_degrees(text)
    text = fix_symbols(text)
    text = squeeze_spaces(text)
    text = fix_split_words(text, vocab, joins)
    p["clean"] = squeeze_spaces(text)

raw_chars = sum(len(p["text"]) for p in pages)
clean_chars = sum(len(p["clean"]) for p in pages)
print(f"Cleaned {len(pages)} pages: {raw_chars:,} -> {clean_chars:,} characters")

Cleaned 60 pages: 128,524 -> 121,227 characters


## Before vs after: heatwave FAQ, page 1

In [7]:
page = next(p for p in pages if p["doc_id"] == "forecast_sop" and p["page"] == 25)
print("BEFORE:\n" + page["text"][:700])
print("\n" + "=" * 60 + "\nAFTER:\n" + page["clean"][:700])

BEFORE:
 
 
SOP for Weather Forecasting and Warning 
10  
 
Table 1.5. 
 
Terminology for intensity of 24 hour accumulated rainfall 
 
S. No. Terminology Rainfall (mm) Rainfall (cm) Percentile 
1. Very light rainfall Trace -2.4   
2. Light rainfall 2.5-15.5 Upto 1 Upto 65 
3. Moderate rainfall 15.6-64.4 2-6 65-95 
4. Heavy Rainfall 64.5- 115.5 7-11 95-99 
5. Very Heavy Rainfall 115.6-204.4 12-20 99.0-99.9 
6. Extremely heavy 
rainfall 
Greater than or 
equal 
to 204.5 mm 
21 cm or 
more 
>99.9 
7. Exceptionally Heavy Rainfall When the rainfall observed is a value near about 
the highest recorded rainfall at or near the station 
for the month or the season. However, this term 
will be used only when 

AFTER:
Table 1.5.

Terminology for intensity of 24 hour accumulated rainfall

S. No. Terminology Rainfall (mm) Rainfall (cm) Percentile
1. Very light rainfall Trace-2.4
2. Light rainfall 2.5-15.5 Upto 1 Upto 65
3. Moderate rainfall 15.6-64.4 2-6 65-95
4. Heavy Rainfall 64.5- 115.5 7-11 95-

### ✅ Check that the cleaning did the right things

In [8]:
all_clean = "\n".join(p["clean"] for p in pages)

print("Broken '0C' left:", len(re.findall(r"\d ?0C\b", all_clean)))
print("Temperatures in the clean text:",
      sorted(set(re.findall(r"\d+(?:\.\d+)?°C", all_clean)), key=lambda s: float(s[:-2])))
print("Headers left:", sum(bool(re.search(p, all_clean, re.M)) for p in HEADER_PATTERNS))
print("Links left:", len(re.findall(r"https?://|%[0-9A-F]{2}", all_clean)))
print("Strange symbols left:", sum(all_clean.count(s) for s in ["", "", "♦", "•", "“", "”"]))
print(f"\nSplit words joined ({sum(joins.values())}):")
print(", ".join(joins))

Broken '0C' left: 0
Temperatures in the clean text: ['0°C', '02°C', '04°C', '4.5°C', '6.4°C', '8°C', '10.0°C', '15.0°C', '30°C', '32°C', '36°C', '37°C', '38°C', '40°C', '45°C', '47°C']
Headers left: 0
Links left: 0
Strange symbols left: 0

Split words joined (99):
te mperature -> temperature, medi cal -> medical, fr om -> from, remai ns -> remains, informatio n -> information, impac t -> impact, increasin g -> increasing, humidit y -> humidity, de clared -> declared, f ollowed -> followed, variatio ns -> variations, publi c -> public, cen tral -> central, co ordination -> coordination, temp erature -> temperature, R elative -> Relative, accompani ed -> accompanied, bet ween -> between, reduc e -> reduce, work ers -> workers, s tandard -> standard, northwa rd -> northward, hemi sphere -> hemisphere, easte rn -> eastern, ove r -> over, norm al -> normal, ma rked -> marked, co ast -> coast, res ponsible -> responsible, enviro nment -> environment, wat ers -> waters, co nditions -> conditi

## Step 4: Cut each document into sections

In [9]:
SECTION_MARKERS = {
    "heat_faq": r"^Q\. ",
    "monsoon_faq": r"^\d{1,2}\. (?:What|How|Why|When|Which|Who|Where|Whether|Is|Are|Does|Do|Can|Will)\b",
    "forecast_sop": r"^\d+\.\d+(?:\.\d+)*\.? [A-Z]",
    "cyclone_terms": r"^[A-Z][A-Za-z ()/,\-]{1,50}\s?:",
}

def join_pages(doc_id):
    text, page_starts = "", []
    for p in pages:
        if p["doc_id"] == doc_id:
            page_starts.append((len(text), p["page"]))
            text += p["clean"] + "\n"
    return text, page_starts

def page_at(position, page_starts):
    page = page_starts[0][1]
    for start, page_no in page_starts:
        if start <= position:
            page = page_no
    return page

def split_sections(doc_id):
    text, page_starts = join_pages(doc_id)
    starts = [m.start() for m in re.finditer(SECTION_MARKERS[doc_id], text, re.M)]
    if not starts or starts[0] > 0:
        starts = [0] + starts
    starts.append(len(text))

    sections = []
    for begin, end in zip(starts, starts[1:]):
        section_text = text[begin:end].strip()
        if not section_text:
            continue
        sections.append({
            "doc_id": doc_id,
            "title": section_text.split("\n")[0][:120],
            "text": section_text,
            "page_start": page_at(begin, page_starts),
            "page_end": page_at(end - 1, page_starts),
        })
    return sections

sections = [s for doc in DOCUMENTS for s in split_sections(doc["doc_id"])]
print(f"{len(sections)} sections in total")

236 sections in total


### How many sections, and how big?

In [10]:
section_table = pd.DataFrame(sections)
section_table["chars"] = section_table["text"].str.len()

section_table.groupby("doc_id")["chars"].agg(
    sections="count",
    smallest="min",
    median="median",
    largest="max",
    too_small=lambda c: (c < 100).sum(),
    too_big=lambda c: (c > 1000).sum(),
)

,sections,smallest,median,largest,too_small,too_big
doc_id,,,,,,
cyclone_terms,107,25,108.0,461,49,0
forecast_sop,29,23,822.0,5822,4,13
heat_faq,42,16,517.5,1736,1,5
monsoon_faq,58,44,666.5,2773,1,21


### 👀 Look at a few sections

In [11]:
for doc_id in ["heat_faq", "monsoon_faq", "forecast_sop", "cyclone_terms"]:
    s = next(s for s in sections if s["doc_id"] == doc_id and 150 < len(s["text"]) < 700)
    print(f"--- {doc_id} | pages {s['page_start']}-{s['page_end']} | {len(s['text'])} chars")
    print(s["text"], "\n")

--- heat_faq | pages 1-1 | 454 chars
Q. What is a heat wave?
Heat Waves are a period of unusually high temperatures as compared to what is
normally expected over a region. Therefore, the temperatures at which Heat waves are
declared differ from place to place based on the temperature climatology (historical
temperatures) of that region. The impact of heat waves gets aggravated by supportive
meteorological factors such as high humidity, high wind speed, duration of heat wave
events, etc. 

--- monsoon_faq | pages 1-1 | 498 chars
1. What is the all India monthly and seasonal rainfall?

All India monthly rainfall is the amount of accumulated rainfall received over India
for a particular month. For example, All India monthly rainfall of June 2018 is
155.7mm. Similarly All India seasonal rainfall is the amount of accumulated rainfall
received over India for a particular season. e.g. All India seasonal rainfall of South-
West monsoon (JJAS) of 2018 is 804.1mm. These quantities are not consta

### 🔍 The problem cases: too big and too small

In [12]:
biggest = section_table.sort_values("chars", ascending=False).head(5)
smallest = section_table.sort_values("chars").head(5)

print("BIGGEST (may mix sub-topics, and the embedding gets blurry):")
for _, s in biggest.iterrows():
    print(f"  {s['chars']:5} chars  [{s['doc_id']}] {s['title'][:70]}")
print("\nSMALLEST (too little meaning on their own):")
for _, s in smallest.iterrows():
    print(f"  {s['chars']:5} chars  [{s['doc_id']}] {s['text'][:70]!r}")

BIGGEST (may mix sub-topics, and the embedding gets blurry):
   5822 chars  [forecast_sop] 7.4. Role and responsibility of different offices of IMD
   4077 chars  [forecast_sop] 7.2. Observational aspects
   2896 chars  [forecast_sop] 7.1. Objective
   2773 chars  [monsoon_faq] 34. How do we declare onset and advance of monsoon?
   2130 chars  [monsoon_faq] 30. What is SEFS?

SMALLEST (too little meaning on their own):
     16 chars  [heat_faq] 'FAQ on Heat Wave'
     23 chars  [forecast_sop] '7.6. Recent development'
     25 chars  [cyclone_terms] 'CMV: Cloud motion vector.'
     25 chars  [cyclone_terms] 'Cyclone: Tropical cyclone'
     25 chars  [cyclone_terms] 'GMT: Greenwich Mean Time.'


## Step 5: Safety rules (too small / too big)

In [13]:
MIN_CHARS, MAX_CHARS, TARGET_CHARS, GLOSSARY_CHARS = 100, 1000, 800, 600

def group_glossary(terms):
    groups, current = [], []
    for term in terms:
        if current and sum(len(t["text"]) for t in current) + len(term["text"]) > GLOSSARY_CHARS:
            groups.append(current)
            current = []
        current.append(term)
    if current:
        groups.append(current)
    return [{
        "doc_id": g[0]["doc_id"],
        "title": f"Glossary: {g[0]['title'].split(':')[0]} ... {g[-1]['title'].split(':')[0]}",
        "text": "\n".join(t["text"] for t in g),
        "page_start": g[0]["page_start"],
        "page_end": g[-1]["page_end"],
    } for g in groups]

def merge_small(doc_sections):
    merged, carry = [], None
    for s in doc_sections:
        if carry:
            s = {**s, "text": carry["text"] + "\n" + s["text"], "page_start": carry["page_start"]}
            carry = None
        if len(s["text"]) < MIN_CHARS:
            carry = s
        else:
            merged.append(s)
    if carry and merged:
        merged[-1] = {**merged[-1], "text": merged[-1]["text"] + "\n" + carry["text"]}
    return merged

def last_sentence(text):
    sentences = re.split(r"(?<=[.?!])\s+", text.strip())
    return sentences[-1] if len(sentences[-1]) < 250 else ""

def split_big(section):
    if len(section["text"]) <= MAX_CHARS:
        return [section]
    body = section["text"][len(section["title"]):].strip()
    paragraphs = [p for p in re.split(r"\n\s*\n", body) if p.strip()]
    pieces = []
    for p in paragraphs:
        pieces += re.split(r"(?<=[.?!])\s+", p) if len(p) > TARGET_CHARS else [p]

    parts, current = [], ""
    for piece in pieces:
        if current and len(current) + len(piece) > TARGET_CHARS:
            parts.append(current)
            current = last_sentence(current)
        current = (current + "\n" + piece).strip()
    parts.append(current)

    n = len(parts)
    return [{**section, "text": f"{section['title']} (part {i}/{n})\n{part}", "part": f"{i}/{n}"}
            for i, part in enumerate(parts, start=1)]

final_sections = []
for doc in DOCUMENTS:
    doc_sections = [s for s in sections if s["doc_id"] == doc["doc_id"]]
    if doc["doc_id"] == "cyclone_terms":
        doc_sections = group_glossary(doc_sections)
    for s in merge_small(doc_sections):
        final_sections += split_big(s)

print(f"{len(sections)} sections -> {len(final_sections)} chunks")

236 sections -> 219 chunks


## Step 6: Add metadata, check, and save

In [14]:
import json

doc_info = {d["doc_id"]: d for d in DOCUMENTS}
chunks = []
for i, s in enumerate(final_sections):
    doc = doc_info[s["doc_id"]]
    chunks.append({
        "chunk_id": f"{s['doc_id']}-{i:03d}",
        "doc_id": s["doc_id"],
        "doc_title": doc["title"],
        "source_url": doc["url"],
        "page_start": s["page_start"],
        "page_end": s["page_end"],
        "section_title": s["title"],
        "text": s["text"],
        "chars": len(s["text"]),
    })

chunk_table = pd.DataFrame(chunks)
chunk_table.groupby("doc_id")["chars"].agg(chunks="count", smallest="min", median="median", largest="max")

,chunks,smallest,median,largest
doc_id,,,,
cyclone_terms,27,201,534.0,599
forecast_sop,56,102,738.0,893
heat_faq,48,189,549.0,886
monsoon_faq,88,199,633.5,972


In [15]:
print("Chunks under 100 characters:", (chunk_table["chars"] < MIN_CHARS).sum())
print("Chunks over 1,100 characters:", (chunk_table["chars"] > 1100).sum())

example = next(c for c in chunks if "(part 2/" in c["text"])
print(f"\nExample of a split chunk ({example['chunk_id']}, pages {example['page_start']}-{example['page_end']}):")
print(example["text"][:500])

Chunks under 100 characters: 0
Chunks over 1,100 characters: 0

Example of a split chunk (heat_faq-026, pages 8-8):
Q. What is impact based heat wave warning issue by India Meteorological (part 2/3)
(b) Wear
lightweight, light-
coloured, loose, cotton
clothes.
(c) Cover your
head: Use a cloth, hat
or umbrella
Orange Alert
(Be
prepared)
Severe
Heat
Alert for
the day
(i) Severe heat
wave conditions
persists for 2 days
(ii) Through not
severe, but heat
wave persists for 4
days or more
High temperature.
Increased likelihood of
heat illness symptoms in
people who are either
exposed to sun for a
prolonged period or


In [16]:
CHUNKS_PATH = Path("../data/chunks.json")
CHUNKS_PATH.parent.mkdir(exist_ok=True)
CHUNKS_PATH.write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved {len(chunks)} chunks to {CHUNKS_PATH}")

Saved 219 chunks to ..\data\chunks.json
